# Mistério em João Pessoa

Adaptação do lendário [SQL Murder Mystery](https://github.com/NUKnightLab/sql-mysteries) (Knight Lab / Northwestern University, conteúdo original sob licença CC BY-SA 4.0). Nesta adaptação você resolve tudo com pandas + MinIO. A cidade e alguns nomes/ruas viraram locais de João Pessoa.

Um assassinato foi registrado em **João Pessoa** em **15/01/2018**. A polícia recolheu 6 arquivos crus (`dados/*.csv`) e é com eles que você vai trabalhar.

## O que você recebeu

| Arquivo | Colunas | O que é |
|---|---|---|
| `ocorrencia.csv` | `data, tipo, descricao, cidade` | Boletins de Ocorrência |
| `pessoa.csv` | `id, nome, detran_id, numero_endereco, rua, cpf` | Cadastro de Pessoas |
| `detran.csv` | `id, idade, altura, cor_olhos, cor_cabelo, genero, placa, marca_veiculo, modelo_veiculo` | Cadastro do DETRAN |
| `depoimento.csv` | `pessoa_id, relato` | Depoimentos |
| `membro_academia.csv` | `id, pessoa_id, nome, data_matricula, plano` | Matrículas da Academia |
| `checkin_academia.csv` | `matricula_id, data_checkin, hora_entrada, hora_saida` | Check-ins da Academia |

## O que você entrega

1. **Fase 1 — Bronze**: os 6 CSVs publicados como tabelas bronze (`df.to_parquet(f"s3://{BUCKET}/bronze/<nome>.parquet", storage_options=STORAGE_OPTIONS)`).
2. **Fase 2 — Investigação**: livre — use pandas (`pd.read_parquet` + `merge`/filtros) para seguir as pistas até chegar a **1** suspeito.
3. **Fase 3 — Resposta final na Silver**: uma tabela `silver.resposta_caso` com sua conclusão (ver Fase 3 no final deste notebook pro formato esperado).

In [4]:
import os

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

BUCKET = os.environ.get("LAKEHOUSE_S3_BUCKET", "lakehouse")
STORAGE_OPTIONS = {
    "key": os.environ.get("LAKEHOUSE_S3_ACCESS_KEY", "trilha"),
    "secret": os.environ.get("LAKEHOUSE_S3_SECRET_KEY", "trilha123"),
    "client_kwargs": {"endpoint_url": os.environ.get("LAKEHOUSE_S3_ENDPOINT", "http://minio:9000")},
}

## Fase 1 — Bronze

O padrão pra publicar qualquer CSV cru como tabela bronze é sempre o mesmo — 2 passos, só pandas:

```python
df = pd.read_csv("dados/<arquivo>.csv")                                                                     # lê o CSV cru direto do disco
df.to_parquet(f"s3://{BUCKET}/bronze/<nome_tabela>.parquet", storage_options=STORAGE_OPTIONS, index=False)  # publica: grava Parquet
```

Cada tabela vira **1 arquivo Parquet** na camada — `bronze/<nome_tabela>.parquet`, dá pra conferir pelo MinIO Console (http://localhost:9001) — os arquivos vão aparecendo em `lakehouse/bronze/`.

Um exemplo pronto (`ocorrencia`), com uma ilustração rápida de como ler de volta logo depois — daí é sua vez de fazer o mesmo padrão para as outras 5 tabelas.

In [5]:
df_ocorrencia = pd.read_csv("dados/ocorrencia.csv")
df_ocorrencia.to_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_ocorrencia

,data,tipo,descricao,cidade
0,20180115,roubo,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,assassinato,Life? Dont talk to me about life.,Albany
2,20180115,assassinato,"Mama, I killed a man, put a gun against his head...",Reno
3,20180215,assassinato,REDACTED REDACTED REDACTED,João Pessoa
4,20180215,assassinato,Someone killed the guard! He took an arrow to the knee!,João Pessoa
...,...,...,...,...
1223,20180430,suborno,\n,Garden Grove
1224,20180430,fraude,‘Why not?’ said the March Hare.\n,Houma
1225,20180430,agressão,\n,Fontana
1226,20180501,agressão,"be NO mistake about it: it was neither more nor less than a pig, and she\n",Trenton


### Lendo de volta (exemplo rápido)

Pra ler uma tabela publicada é só apontar o `pd.read_parquet` pro mesmo caminho, ilustrado com a tabela que acabamos de publicar:

- `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)` lê o Parquet inteiro direto do MinIO com pandas — é o que você vai usar na Fase 2 pra ler cada uma das 6 tabelas bronze.

Pra ver o que já existe fisicamente numa camada, sem precisar ler o conteúdo nem escrever código nenhum: abra o **MinIO Console** (http://localhost:9001) e olhe os arquivos em `lakehouse/bronze/`.

In [6]:
# Lendo a tabela de volta direto do MinIO com pandas — deve ser idêntica a df_ocorrencia
pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)

,data,tipo,descricao,cidade
0,20180115,roubo,A Man Dressed as Spider-Man Is on a Robbery Spree,NYC
1,20180115,assassinato,Life? Dont talk to me about life.,Albany
2,20180115,assassinato,"Mama, I killed a man, put a gun against his head...",Reno
3,20180215,assassinato,REDACTED REDACTED REDACTED,João Pessoa
4,20180215,assassinato,Someone killed the guard! He took an arrow to the knee!,João Pessoa
...,...,...,...,...
1223,20180430,suborno,\n,Garden Grove
1224,20180430,fraude,‘Why not?’ said the March Hare.\n,Houma
1225,20180430,agressão,\n,Fontana
1226,20180501,agressão,"be NO mistake about it: it was neither more nor less than a pig, and she\n",Trenton


In [7]:
df_pessoa = pd.read_csv("dados/pessoa.csv")
df_pessoa.to_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_pessoa

,id,nome,detran_id,numero_endereco,rua,cpf
0,10000,Christoper Peteuil,993845,624,Bankhall Ave,747714076
1,10007,Kourtney Calderwood,861794,2791,Gustavus Blvd,477972044
2,10010,Muoi Cary,385336,741,Avenida Ministro José Américo de Almeida,828638512
3,10016,Era Moselle,431897,1987,Wood Glade St,614621061
4,10025,Trena Hornby,550890,276,Daws Hill Way,223877684
...,...,...,...,...,...,...
10006,99936,Luba Benser,274427,680,Carnage Blvd,685095054
10007,99941,Roxana Mckimley,975942,1613,Gate St,512136801
10008,99965,Cherie Zeimantz,287627,3661,The Water Ave,362877324
10009,99982,Allen Cruse,251350,3126,N Jean Dr,348734531


In [8]:
# detran.csv -> bronze.detran
df_detran = pd.read_csv("dados/detran.csv")
df_detran.to_parquet(f"s3://{BUCKET}/bronze/detran.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_detran

,id,idade,altura,cor_olhos,cor_cabelo,genero,placa,marca_veiculo,modelo_veiculo
0,100280,72,57,castanho,ruivo,masculino,P24L4U,Acura,MDX
1,100460,63,72,castanho,castanho,feminino,XF02T6,Cadillac,SRX
2,101029,62,74,verde,verde,feminino,VKY5KR,Scion,xB
3,101198,43,54,mel,castanho,feminino,Y5NZ08,Nissan,Rogue
4,101255,18,79,azul,grisalho,feminino,5162Z1,Lexus,GS
...,...,...,...,...,...,...,...,...,...
10002,999923,19,77,mel,preto,feminino,5L0ZI4,GMC,Sierra 3500
10003,999940,71,61,verde,verde,masculino,1B8QN8,Mitsubishi,Eclipse
10004,999981,67,69,castanho,azul,feminino,1684K3,Land Rover,LR2
10005,999986,49,58,verde,grisalho,masculino,F8F64H,Lexus,LS


In [9]:
# depoimento.csv -> bronze.depoimento
df_depoimento = pd.read_csv("dados/depoimento.csv")
df_depoimento.to_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_depoimento

,pessoa_id,relato
0,28508,‘I deny it!’ said the March Hare.\n
1,63713,\n
2,86208,"way, and the whole party swam to the shore.\n"
3,35267,"lessons in here? Why, there’s hardly room for YOU, and no room at all\n"
4,33856,\n
...,...,...
4986,37357,"Alice did not wish to offend the Dormouse again, so she began very\n"
4987,10206,"time,’ she said, ‘than waste it in asking riddles that have no answers.’\n"
4988,14887,"Eu ouvi um tiro e depois vi um homem saindo correndo. Ele tinha uma bolsa da ""Academia Kongo"". O número da matrícula na bolsa começava com ""48Z"". Só sócios do plano ouro têm essas bolsas. O homem entrou num carro com uma placa que continha ""H42W""."
4989,16371,"Eu vi o assassinato acontecer, e reconheci o assassino da minha academia, de quando eu estava treinando na semana passada, no dia 9 de janeiro."


In [10]:
# membro_academia.csv -> bronze.membro_academia
df_membro_academia = pd.read_csv("dados/membro_academia.csv")
df_membro_academia.to_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_membro_academia

,id,pessoa_id,nome,data_matricula,plano
0,NL318,65076,Everette Koepke,20170926,ouro
1,AOE21,39426,Noe Locascio,20171005,regular
2,2PN28,63823,Jeromy Heitschmidt,20180215,prata
3,0YJ24,80651,Waneta Wellard,20171206,ouro
4,3A08L,32858,Mei Bianchin,20170401,prata
...,...,...,...,...,...
179,2V137,41693,Wendell Dulany,20171219,prata
180,4KB72,79110,Emile Hege,20170522,regular
181,48Z7A,28819,Severino Bezerra Lima,20160305,ouro
182,48Z55,67318,Genildo Cavalcanti Farias,20160101,ouro


In [11]:
# checkin_academia.csv -> bronze.checkin_academia
df_checkin_academia = pd.read_csv("dados/checkin_academia.csv")
df_checkin_academia.to_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_checkin_academia

,matricula_id,data_checkin,hora_entrada,hora_saida
0,NL318,20180212,329,365
1,NL318,20170811,469,920
2,NL318,20180429,506,554
3,NL318,20180128,124,759
4,NL318,20171027,418,1019
...,...,...,...,...
2698,4KB72,20170422,1016,1114
2699,4KB72,20170630,408,885
2700,48Z7A,20180109,1600,1730
2701,48Z55,20180109,1530,1700


Checagem: as 6 tabelas devem aparecer em `lakehouse/bronze/` no **MinIO Console** (http://localhost:9001).

## Fase 2 — Investigação (pandas)

A partir de agora é livre: leia as tabelas bronze direto do MinIO com `pd.read_parquet(f"s3://{BUCKET}/bronze/<tabela>.parquet", storage_options=STORAGE_OPTIONS)`, e siga as pistas com `merge`/filtros de DataFrame, do jeito que preferir.

Um roteiro sugerido (não obrigatório seguir exatamente esta ordem, mas ajuda a não se perder):

1. Pesquise a ocorrência e bus as testemunhas em `pessoa`.
2. Pesquise o `depoimento` das respectivas testemunhas.
3. Cada depoimento traz uma pista diferente — uma aponta para `membro_academia` a outra para `detran`.
4. Pesquise pela pistas, mencionadas pelas testemunhas, nas tabelas correspondentes.
5. Confirme em `checkin_academia` que o suspeito tem um check-in na academia na data que a segunda testemunha mencionou.

In [12]:
# Lendo todas as tabelas bronze direto do MinIO via pandas
ocorrencia = pd.read_parquet(f"s3://{BUCKET}/bronze/ocorrencia.parquet", storage_options=STORAGE_OPTIONS)
pessoa = pd.read_parquet(f"s3://{BUCKET}/bronze/pessoa.parquet", storage_options=STORAGE_OPTIONS)
detran = pd.read_parquet(f"s3://{BUCKET}/bronze/detran.parquet", storage_options=STORAGE_OPTIONS)
depoimento = pd.read_parquet(f"s3://{BUCKET}/bronze/depoimento.parquet", storage_options=STORAGE_OPTIONS)
membro_academia = pd.read_parquet(f"s3://{BUCKET}/bronze/membro_academia.parquet", storage_options=STORAGE_OPTIONS)
checkin_academia = pd.read_parquet(f"s3://{BUCKET}/bronze/checkin_academia.parquet", storage_options=STORAGE_OPTIONS)

### Passo 1 — a ocorrência

Um assassinato foi registrado em **João Pessoa** em **15/01/2018**.

In [13]:
# O assassinato em João Pessoa, em 15/01/2018
crime = ocorrencia[
    (ocorrencia["data"] == 20180115)
    & (ocorrencia["tipo"] == "assassinato")
    & (ocorrencia["cidade"] == "João Pessoa")
]
crime

,data,tipo,descricao,cidade
1227,20180115,assassinato,"As imagens de segurança mostram que houve 2 testemunhas. A primeira testemunha mora na última casa da ""Avenida Ministro José Américo de Almeida"". A segunda testemunha, chamada Maria Aparecida, mora em algum lugar da ""Avenida Rui Carneiro"".",João Pessoa


### Passo 2 — as testemunhas

In [14]:
# A descrição da ocorrência aponta 2 testemunhas:
# - a 1a mora na última casa da "Avenida Ministro José Américo de Almeida" (maior numero_endereco)
# - a 2a se chama Maria Aparecida (Nunes) e mora na "Avenida Rui Carneiro"
testemunha_1 = (
    pessoa[pessoa["rua"] == "Avenida Ministro José Américo de Almeida"]
    .sort_values("numero_endereco", ascending=False)
    .head(1)
)
testemunha_2 = pessoa[
    (pessoa["rua"] == "Avenida Rui Carneiro") & (pessoa["nome"] == "Maria Aparecida Nunes")
]
testemunhas = pd.concat([testemunha_1, testemunha_2])
testemunhas

,id,nome,detran_id,numero_endereco,rua,cpf
499,14887,Josenildo Pereira da Rocha,118009,4919,Avenida Ministro José Américo de Almeida,111564949
665,16371,Maria Aparecida Nunes,490173,103,Avenida Rui Carneiro,318771143


### Passo 3 — duas pistas, duas tabelas


In [15]:
# Depoimentos das duas testemunhas
depoimentos_testemunhas = depoimento[depoimento["pessoa_id"].isin(testemunhas["id"])]
depoimentos_testemunhas

,pessoa_id,relato
4988,14887,"Eu ouvi um tiro e depois vi um homem saindo correndo. Ele tinha uma bolsa da ""Academia Kongo"". O número da matrícula na bolsa começava com ""48Z"". Só sócios do plano ouro têm essas bolsas. O homem entrou num carro com uma placa que continha ""H42W""."
4989,16371,"Eu vi o assassinato acontecer, e reconheci o assassino da minha academia, de quando eu estava treinando na semana passada, no dia 9 de janeiro."


In [16]:
# Pista da testemunha 1: matrícula de academia começando com "48Z", plano ouro,
# e placa de carro contendo "H42W"
pista_academia = membro_academia[
    membro_academia["id"].str.startswith("48Z") & (membro_academia["plano"] == "ouro")
]
pista_veiculo = detran[detran["placa"].str.contains("H42W")]
pista_academia, pista_veiculo

(        id  pessoa_id                       nome  data_matricula plano
 181  48Z7A      28819      Severino Bezerra Lima        20160305  ouro
 182  48Z55      67318  Genildo Cavalcanti Farias        20160101  ouro,
           id  idade  altura cor_olhos cor_cabelo     genero   placa  \
 915   183779     21      65      azul      loiro   feminino  H42W0X   
 3529  423327     30      70  castanho   castanho  masculino  0H42W2   
 6240  664760     21      71     preto      preto  masculino  4H42WR   
 
      marca_veiculo modelo_veiculo  
 915           Fiat            Uno  
 3529          Jeep       Renegade  
 6240    Volkswagen          Fusca  )

### Passo 4 — cruzando as pistas

In [17]:
# Cruzando as duas pistas: quem aparece nas duas ao mesmo tempo (mesma pessoa)?
suspeito = pessoa[
    pessoa["id"].isin(pista_academia["pessoa_id"])
    & pessoa["detran_id"].isin(pista_veiculo["id"])
]
suspeito

,id,nome,detran_id,numero_endereco,rua,cpf
6327,67318,Genildo Cavalcanti Farias,423327,530,"Washington Pl, Apt 3A",871539279


### Passo 5 — confirmar com o check-in

In [18]:
# Testemunha 2 disse ter reconhecido o assassino da academia, treinando no dia 09/01/2018.
# Confirma que a matrícula do suspeito tem check-in nessa data.
matricula_suspeito = pista_academia.loc[
    pista_academia["pessoa_id"].isin(suspeito["id"]), "id"
].iloc[0]

checkin_academia[
    (checkin_academia["matricula_id"] == matricula_suspeito)
    & (checkin_academia["data_checkin"] == 20180109)
]

,matricula_id,data_checkin,hora_entrada,hora_saida
2701,48Z55,20180109,1530,1700


## Fase 3 — Resposta final na Silver

Chegou a hora de publicar sua conclusão como uma tabela — o entregável desta tarefa. Monte um DataFrame de **1 linha** com estas colunas:

| coluna | conteúdo |
|---|---|
| `nome_suspeito` | o nome completo da pessoa em `pessoa` |
| `placa_veiculo` | a placa (de `detran`) que fechou o caso |
| `pista_academia` | qual detalhe da matrícula (início do `id` + status do plano) bateu com o depoimento |
| `pista_veiculo` | qual trecho da placa bateu com o depoimento |
| `justificativa` | 1-2 frases explicando o raciocínio (pode ser texto livre) |

E publique com `df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)` — depois disso, o **MinIO Console** (http://localhost:9001), em `lakehouse/silver/resposta_caso.parquet`, já mostra o resultado.

In [19]:
placa_suspeito = pista_veiculo.loc[
    pista_veiculo["id"] == suspeito["detran_id"].iloc[0], "placa"
].iloc[0]

df_resposta = pd.DataFrame([{
    "nome_suspeito": suspeito["nome"].iloc[0],
    "placa_veiculo": placa_suspeito,
    "pista_academia": f"matrícula {matricula_suspeito} (começa com '48Z'), plano ouro",
    "pista_veiculo": "placa contém 'H42W'",
    "justificativa": (
        "As duas testemunhas do assassinato em João Pessoa (15/01/2018) deram pistas "
        "complementares: matrícula de academia começando com '48Z' , plano ouro, e placa "
        "contendo 'H42W'. Só Genildo Cavalcanti Farias bate nas duas ao mesmo tempo, "
        "e seu check-in na Academia Kongo em 09/01/2018 confirma o relato da segunda "
        "testemunha, que disse ter reconhecido ele treinando nessa data."
    ),
}])
df_resposta.to_parquet(f"s3://{BUCKET}/silver/resposta_caso.parquet", storage_options=STORAGE_OPTIONS, index=False)
df_resposta

,nome_suspeito,placa_veiculo,pista_academia,pista_veiculo,justificativa
0,Genildo Cavalcanti Farias,0H42W2,"matrícula 48Z55 (começa com '48Z'), plano ouro",placa contém 'H42W',"As duas testemunhas do assassinato em João Pessoa (15/01/2018) deram pistas complementares: matrícula de academia começando com '48Z' , plano ouro, e placa contendo 'H42W'. Só Genildo Cavalcanti Farias bate nas duas ao mesmo tempo, e seu check-in na Academia Kongo em 09/01/2018 confirma o relato da segunda testemunha, que disse ter reconhecido ele treinando nessa data."


---

Terminou? `lakehouse/bronze/` deve ter as 6 tabelas desta tarefa, e `lakehouse/silver/resposta_caso.parquet` deve ter sua conclusão — dá pra confirmar tudo pelo MinIO Console (http://localhost:9001) sem precisar de mais nada.